# ECG Analysis — DACIL-WESENSE

This notebook extracts time-resolved ECG features from the BDF files (WESENSE L1 / L2 sensors) and analyses them at both the individual-patient and cohort level.

**Features extracted per 30-second window:**
- Heart rate (bpm)
- HRV time-domain: RMSSD and SDNN (ms)
- HRV frequency-domain: LF, HF power (ms²) and LF/HF ratio
- Breathing rate via ECG-derived respiration (EDR, bpm)

> **Note on frequency-domain HRV**: clinical guidelines (Task Force 1996) recommend ≥ 5-minute windows for LF/HF analysis. At the default 30-second window these values are exploratory only.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import functions as fn

# ── Configuration ──────────────────────────────────────────────────────────
DATA_ROOT   = "data"
OUTPUT_ROOT = "output"

# Set to a specific patient folder name (e.g. "Patient 1") to pin the
# single-patient deep-dive, or leave as None to use the first available.
PATIENT_ID: str | None = None

# Analysis window length in seconds (default 30 s).
WINDOW_DURATION: float = 30.0

In [ ]:
fn.setup_logging(log_file="pipeline.log")

---
## Part A — Single-patient deep-dive

In [ ]:
# ── Discover patient folders ───────────────────────────────────────────────
all_folders = fn.discover_patient_folders(DATA_ROOT)
if not all_folders:
    raise RuntimeError(f"No patient folders found under '{DATA_ROOT}'")

if PATIENT_ID is not None:
    folder = next((f for f in all_folders if f.name == PATIENT_ID), None)
    if folder is None:
        raise ValueError(f"Patient '{PATIENT_ID}' not found in '{DATA_ROOT}'")
else:
    folder = all_folders[0]
    PATIENT_ID = folder.name

print(f"Single-patient analysis: {PATIENT_ID}")

# ── Load BDF files ─────────────────────────────────────────────────────────
l1_path, l2_path = fn.find_bdf_files(folder)

raw_l1 = fn.load_ecg(l1_path) if l1_path is not None else None
raw_l2 = fn.load_ecg(l2_path) if l2_path is not None else None

if raw_l1 is None and raw_l2 is None:
    raise RuntimeError(f"No BDF files found or loadable for patient '{PATIENT_ID}'")

print(f"L1: {'loaded' if raw_l1 is not None else 'not available'}")
print(f"L2: {'loaded' if raw_l2 is not None else 'not available'}")

In [ ]:
# ── Extract time-resolved ECG features ────────────────────────────────────
ts_frames = []

for sensor_label, raw in [("L1", raw_l1), ("L2", raw_l2)]:
    if raw is None:
        continue
    print(f"Extracting ECG timeseries for sensor {sensor_label} …")
    ts = fn.extract_ecg_timeseries(raw, window_duration=WINDOW_DURATION)
    ts["sensor"] = sensor_label
    ts_frames.append(ts)

ts_all = pd.concat(ts_frames, ignore_index=True)
print(f"\nTimeseries shape: {ts_all.shape}")
ts_all.head()

### Heart rate over time

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

for (sensor, channel), grp in ts_all.groupby(["sensor", "channel"]):
    label = f"{sensor} – {channel}"
    ax.plot(grp["time_s"] / 60, grp["hr_bpm"], label=label, linewidth=1.2)
    mean_hr = grp["hr_bpm"].mean()
    if not np.isnan(mean_hr):
        ax.axhline(mean_hr, linestyle="--", linewidth=0.8, alpha=0.6)

ax.set_xlabel("Time (min)")
ax.set_ylabel("Heart rate (bpm)")
ax.set_title(f"{PATIENT_ID} — Heart rate over time")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()

out_dir = Path(OUTPUT_ROOT) / PATIENT_ID
fn._save_figure(fig, out_dir, f"{PATIENT_ID}_ecg_hr.png")
plt.show()

### HRV — time-domain (RMSSD & SDNN)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

for (sensor, channel), grp in ts_all.groupby(["sensor", "channel"]):
    label = f"{sensor} – {channel}"
    t = grp["time_s"] / 60
    axes[0].plot(t, grp["rmssd_ms"], label=label, linewidth=1.2)
    axes[1].plot(t, grp["sdnn_ms"],  label=label, linewidth=1.2)

axes[0].set_ylabel("RMSSD (ms)")
axes[0].set_title(f"{PATIENT_ID} — HRV time-domain")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_ylabel("SDNN (ms)")
axes[1].set_xlabel("Time (min)")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
fn._save_figure(fig, out_dir, f"{PATIENT_ID}_ecg_hrv_timedomain.png")
plt.show()

### HRV — frequency-domain (LF, HF, LF/HF)

> These values are computed with 30-second windows. Interpret LF/HF cautiously — the clinical standard requires ≥ 5-minute windows.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

for (sensor, channel), grp in ts_all.groupby(["sensor", "channel"]):
    label = f"{sensor} – {channel}"
    t = grp["time_s"] / 60
    axes[0].plot(t, grp["lf_ms2"],      label=label, linewidth=1.2)
    axes[1].plot(t, grp["hf_ms2"],      label=label, linewidth=1.2)
    axes[2].plot(t, grp["lf_hf_ratio"], label=label, linewidth=1.2)

axes[0].set_ylabel("LF power (ms²)")
axes[0].set_title(f"{PATIENT_ID} — HRV frequency-domain (exploratory at 30 s windows)")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_ylabel("HF power (ms²)")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

axes[2].set_ylabel("LF/HF ratio")
axes[2].set_xlabel("Time (min)")
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

fig.tight_layout()
fn._save_figure(fig, out_dir, f"{PATIENT_ID}_ecg_hrv_freqdomain.png")
plt.show()

### Breathing rate over time (ECG-derived respiration)

> The breathing rate is derived from R-peak amplitude modulation (EDR method) and is an approximation. Cross-validate against VE from telemetry where available.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

for (sensor, channel), grp in ts_all.groupby(["sensor", "channel"]):
    label = f"{sensor} – {channel} (EDR)"
    ax.plot(grp["time_s"] / 60, grp["breathing_rate_bpm"], label=label, linewidth=1.2)

ax.set_xlabel("Time (min)")
ax.set_ylabel("Breathing rate (breaths/min, EDR)")
ax.set_title(f"{PATIENT_ID} — Breathing rate over time")
ax.set_ylim(0, 40)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()

fn._save_figure(fig, out_dir, f"{PATIENT_ID}_ecg_breathing.png")
plt.show()

---
## Part B — Batch summary across all patients

In [ ]:
summary_rows = []

for fold in all_folders:
    pid = fold.name
    l1, l2 = fn.find_bdf_files(fold)

    frames = []
    for sensor_label, bdf_path in [("L1", l1), ("L2", l2)]:
        if bdf_path is None:
            continue
        raw = fn.load_ecg(bdf_path)
        if raw is None:
            continue
        ts = fn.extract_ecg_timeseries(raw, window_duration=WINDOW_DURATION)
        ts["sensor"] = sensor_label
        frames.append(ts)

    if not frames:
        print(f"  {pid}: no BDF data — skipping")
        continue

    ts_pat = pd.concat(frames, ignore_index=True)

    summary_rows.append(
        dict(
            patient_id=pid,
            mean_hr_bpm=ts_pat["hr_bpm"].mean(),
            mean_rmssd_ms=ts_pat["rmssd_ms"].mean(),
            mean_sdnn_ms=ts_pat["sdnn_ms"].mean(),
            mean_lf_ms2=ts_pat["lf_ms2"].mean(),
            mean_hf_ms2=ts_pat["hf_ms2"].mean(),
            mean_lf_hf=ts_pat["lf_hf_ratio"].mean(),
            mean_breathing_bpm=ts_pat["breathing_rate_bpm"].mean(),
        )
    )
    print(f"  {pid}: done")

batch_df = pd.DataFrame(summary_rows)
print(f"\nBatch summary: {len(batch_df)} patients")
batch_df

In [ ]:
import matplotlib.ticker as mticker

metrics = [
    ("mean_hr_bpm",       "Mean HR (bpm)"),
    ("mean_rmssd_ms",     "Mean RMSSD (ms)"),
    ("mean_sdnn_ms",      "Mean SDNN (ms)"),
    ("mean_lf_ms2",       "Mean LF power (ms²)"),
    ("mean_hf_ms2",       "Mean HF power (ms²)"),
    ("mean_lf_hf",        "Mean LF/HF ratio"),
    ("mean_breathing_bpm", "Mean breathing rate (bpm, EDR)"),
]

n_metrics = len(metrics)
fig, axes = plt.subplots(1, n_metrics, figsize=(4 * n_metrics, 5))

for ax, (col, ylabel) in zip(axes, metrics):
    data = batch_df[col].dropna()
    ax.boxplot(data, widths=0.5)
    ax.scatter([1] * len(data), data, color="steelblue", s=30, zorder=3, alpha=0.7)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_xticks([])
    ax.grid(True, alpha=0.3, axis="y")

fig.suptitle("ECG feature summary — all patients", fontsize=12)
fig.tight_layout()

batch_out_dir = Path(OUTPUT_ROOT)
batch_out_dir.mkdir(parents=True, exist_ok=True)
fn._save_figure(fig, batch_out_dir, "ecg_batch_summary.png")
plt.show()